# GymRAVANA model explainability and export

This notebook explains the candidate selected by Notebook 02 and creates the final local artifact package. It is deliberately fail-closed: without a dataset-bound selection report produced from genuine, sufficient labels, it exports nothing.

## Preconditions

Run the current exporter, review Notebook 01, and run Notebook 02 first. An administrator or supervisor must review the held-out and grouped cross-validation evidence before treating the selected candidate as academically defensible. Passing software gates does not prove fairness, safety, causality or suitability for autonomous decisions.

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json

import joblib
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
working_directory = Path.cwd().resolve()
if working_directory.name == 'notebooks':
    PROJECT_ROOT = working_directory.parent.parent
elif (working_directory / 'ai').is_dir():
    PROJECT_ROOT = working_directory
else:
    raise RuntimeError('Open this notebook from the GymRAVANA project or ai/notebooks directory.')

DATASET_PATH = PROJECT_ROOT / 'ai' / 'data' / 'readiness_dataset.csv'
METADATA_PATH = PROJECT_ROOT / 'ai' / 'data' / 'readiness_dataset.metadata.json'
ARTIFACTS_DIR = PROJECT_ROOT / 'ai' / 'artifacts'
SELECTION_REPORT_PATH = ARTIFACTS_DIR / 'model_selection.json'
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError('Dataset not found. Run: php artisan gymravana:export-readiness-data')
if not METADATA_PATH.exists():
    raise FileNotFoundError('Dataset metadata not found. Export the dataset again.')

df = pd.read_csv(DATASET_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
actual_hash = sha256(DATASET_PATH.read_bytes()).hexdigest()
assert metadata.get('schema_version') == 1, 'Unsupported readiness dataset schema.'
assert metadata.get('dataset_sha256') == actual_hash, 'CSV fingerprint does not match its metadata.'
assert metadata.get('row_count') == len(df), 'Metadata row count does not match the CSV.'
assert metadata.get('columns') == list(df.columns), 'Metadata columns do not match the CSV header.'
assert metadata.get('target') == 'ready_for_progression', 'Unexpected target in dataset metadata.'
print({'rows': len(df), 'dataset_sha256': actual_hash})

In [ ]:
NUMERIC_FEATURES = [
    'workout_completions', 'wellness_completions',
    'trainer_sessions_scheduled', 'trainer_sessions_completed',
    'attendance_rate', 'cancelled_or_declined_sessions',
    'active_days', 'consistency_rate', 'activity_points',
    'previous_goal_completion', 'previous_rating',
    'workout_change', 'consistency_change',
]
CATEGORICAL_FEATURES = ['previous_assessment']
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = 'ready_for_progression'
GROUP_COLUMN = 'member_key'
AUDIT_ONLY_COLUMNS = ['observation_month', 'label_recorded_at']
REQUIRED_COLUMNS = set(MODEL_FEATURES + [TARGET, GROUP_COLUMN] + AUDIT_ONLY_COLUMNS)
FORBIDDEN_COLUMNS = {
    'user_id', 'trainer_profile_id', 'name', 'email', 'phone',
    'weight_kg', 'height_cm', 'waist_cm', 'chest_cm',
    'trainer_notes', 'readiness_rationale', 'therapy_request', 'diagnosis',
}

missing_columns = sorted(REQUIRED_COLUMNS - set(df.columns))
forbidden_columns = sorted(set(df.columns) & FORBIDDEN_COLUMNS)
assert not missing_columns, f'Missing required columns: {missing_columns}'
assert not forbidden_columns, f'Sensitive or leakage-prone columns found: {forbidden_columns}'
for column in NUMERIC_FEATURES + [TARGET]:
    df[column] = pd.to_numeric(df[column], errors='coerce')
observed_targets = set(df[TARGET].dropna().astype(int).unique().tolist())
assert observed_targets <= {0, 1}, f'Unexpected target values: {sorted(observed_targets)}'
if not df.empty:
    assert df[TARGET].notna().all(), 'Every exported row must have a readiness label.'
    assert df[GROUP_COLUMN].notna().all(), 'Every row needs a pseudonymous member group.'
duplicate_count = int(df.duplicated(subset=[GROUP_COLUMN, 'observation_month', 'label_recorded_at']).sum()) if not df.empty else 0
conflicting_member_months = int((df.groupby([GROUP_COLUMN, 'observation_month'])[TARGET].nunique() > 1).sum()) if not df.empty else 0
assert duplicate_count == 0, f'Duplicate observation keys found: {duplicate_count}'
assert conflicting_member_months == 0, 'Contradictory readiness labels exist for the same member and observation month.'
print({'targets': sorted(observed_targets), 'duplicates': duplicate_count, 'conflicting_member_months': conflicting_member_months})

In [ ]:
selection_report = None
export_allowed = False
block_reasons = []

if not SELECTION_REPORT_PATH.exists():
    block_reasons.append('Notebook 02 has not produced model_selection.json')
else:
    selection_report = json.loads(SELECTION_REPORT_PATH.read_text(encoding='utf-8'))
    assert selection_report.get('report_version') == 1, 'Unsupported model-selection report.'
    assert selection_report.get('dataset_schema_version') == metadata['schema_version'], 'Selection report schema mismatch.'
    assert selection_report.get('dataset_sha256') == actual_hash, 'Selection report belongs to a different dataset export.'
    assert selection_report.get('dataset_rows') == len(df), 'Selection report row count mismatch.'
    assert selection_report.get('model_features') == MODEL_FEATURES, 'Selection report feature order mismatch.'
    assert selection_report.get('selected_model') in {'logistic_regression', 'random_forest'}, 'Unsupported selected model.'
    if not selection_report.get('gate_report', {}).get('training_allowed'):
        block_reasons.append('Notebook 02 training gate did not pass')
    if not selection_report.get('holdout_results'):
        block_reasons.append('held-out evaluation results are missing')
    if not selection_report.get('cross_validation_results'):
        block_reasons.append('grouped cross-validation results are missing')

export_allowed = selection_report is not None and not block_reasons
print(json.dumps({'export_allowed': export_allowed, 'reasons': block_reasons}, indent=2))

## Explainability method

Permutation importance is used on the untouched grouped holdout because it works consistently for both candidate pipelines and reports importance in the original feature space. Local sensitivity examples compare a prediction with one feature replaced by a training-set baseline. These are descriptive model-behaviour checks, not causal explanations. SHAP is deliberately deferred: adding it is not justified for an empty or minimum-size dataset, and it is not required to make the deterministic baselines transparent.

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, NUMERIC_FEATURES),
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
])
candidate_estimators = {
    'logistic_regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(
        n_estimators=200, min_samples_leaf=2, class_weight='balanced_subsample',
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

selected_pipeline = None
train_df = pd.DataFrame(columns=df.columns)
test_df = pd.DataFrame(columns=df.columns)
if export_allowed:
    y = df[TARGET].astype(int)
    splitter = GroupShuffleSplit(n_splits=20, test_size=0.2, random_state=RANDOM_STATE)
    selected_indices = None
    for train_index, test_index in splitter.split(df[MODEL_FEATURES], y, groups=df[GROUP_COLUMN]):
        if set(y.iloc[train_index].unique()) == {0, 1} and set(y.iloc[test_index].unique()) == {0, 1}:
            selected_indices = (train_index, test_index)
            break
    assert selected_indices is not None, 'The deterministic grouped holdout can no longer be reproduced.'
    train_index, test_index = selected_indices
    train_df = df.iloc[train_index].copy()
    test_df = df.iloc[test_index].copy()
    assert not (set(train_df[GROUP_COLUMN]) & set(test_df[GROUP_COLUMN])), 'Member leakage detected.'
    selected_pipeline = Pipeline([
        ('preprocessing', clone(preprocessor)),
        ('classifier', clone(candidate_estimators[selection_report['selected_model']])),
    ])
    selected_pipeline.fit(train_df[MODEL_FEATURES], train_df[TARGET].astype(int))
    print({'selected_model': selection_report['selected_model'], 'train_rows': len(train_df), 'test_rows': len(test_df)})
else:
    print('Candidate reconstruction skipped; no model was fitted.')

In [ ]:
importance_table = pd.DataFrame(columns=['feature', 'importance_mean', 'importance_std'])
if export_allowed:
    importance = permutation_importance(
        selected_pipeline, test_df[MODEL_FEATURES], test_df[TARGET].astype(int),
        scoring='f1', n_repeats=20, random_state=RANDOM_STATE, n_jobs=1,
    )
    importance_table = pd.DataFrame({
        'feature': MODEL_FEATURES,
        'importance_mean': importance.importances_mean,
        'importance_std': importance.importances_std,
    }).sort_values('importance_mean', ascending=False)
    display(importance_table)
else:
    print('Global explanation skipped because export prerequisites did not pass.')

In [ ]:
local_explanations = []
if export_allowed:
    baselines = {feature: train_df[feature].median() for feature in NUMERIC_FEATURES}
    baselines['previous_assessment'] = (
        train_df['previous_assessment'].mode(dropna=True).iloc[0]
        if not train_df['previous_assessment'].mode(dropna=True).empty else 'unknown'
    )
    for example_number, row_index in enumerate(test_df.head(3).index, start=1):
        original = test_df.loc[[row_index], MODEL_FEATURES].copy()
        probability = float(selected_pipeline.predict_proba(original)[0, 1])
        sensitivities = []
        for feature in MODEL_FEATURES:
            changed = original.copy()
            changed[feature] = baselines[feature]
            baseline_probability = float(selected_pipeline.predict_proba(changed)[0, 1])
            sensitivities.append({
                'feature': feature,
                'probability_change_when_replaced_by_baseline': probability - baseline_probability,
            })
        local_explanations.append({
            'example': example_number,
            'predicted_ready_probability': probability,
            'predicted_class': int(probability >= 0.5),
            'actual_class': int(test_df.at[row_index, TARGET]),
            'strongest_sensitivities': sorted(sensitivities, key=lambda item: abs(item['probability_change_when_replaced_by_baseline']), reverse=True)[:5],
        })
    print(json.dumps(local_explanations, indent=2))
else:
    print('Local explanation examples skipped because no candidate was fitted.')

## Required interpretation and limitations

Feature importance describes this fitted model, not the causes of human readiness. Correlated features can divide or obscure importance. Local sensitivity is not a counterfactual promise. Trainer labels can contain judgment bias, and a small GymRAVANA sample may not represent future members. The model excludes medical data and must never diagnose injury or replace trainer/Master review. A prediction is one auditable Master Gate input, never the sole decision.

In [ ]:
exported_paths = []
if export_allowed:
    final_pipeline = Pipeline([
        ('preprocessing', clone(preprocessor)),
        ('classifier', clone(candidate_estimators[selection_report['selected_model']])),
    ])
    final_pipeline.fit(df[MODEL_FEATURES], df[TARGET].astype(int))
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

    model_path = ARTIFACTS_DIR / 'readiness_model.joblib'
    schema_path = ARTIFACTS_DIR / 'feature_schema.json'
    metrics_path = ARTIFACTS_DIR / 'model_metrics.json'
    metadata_path = ARTIFACTS_DIR / 'model_metadata.json'
    importance_path = ARTIFACTS_DIR / 'feature_importance.json'

    joblib.dump(final_pipeline, model_path)
    feature_schema = {
        'schema_version': 1,
        'features': [
            *[{'name': feature, 'type': 'number', 'nullable': True} for feature in NUMERIC_FEATURES],
            {'name': 'previous_assessment', 'type': 'string', 'nullable': True},
        ],
        'feature_order': MODEL_FEATURES,
        'target': TARGET,
    }
    metrics = {
        'selection_rule': selection_report['selection_rule'],
        'holdout_results': selection_report['holdout_results'],
        'cross_validation_results': selection_report['cross_validation_results'],
    }
    importance_records = json.loads(importance_table.to_json(orient='records'))
    model_metadata = {
        'artifact_version': 1,
        'model_name': selection_report['selected_model'],
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'dataset_schema_version': metadata['schema_version'],
        'dataset_sha256': actual_hash,
        'training_rows': int(len(df)),
        'class_counts': metadata.get('label_counts', {}),
        'random_state': RANDOM_STATE,
        'decision_threshold': 0.5,
        'scikit_learn_version': sklearn.__version__,
        'model_sha256': sha256(model_path.read_bytes()).hexdigest(),
        'intended_use': 'Advisory non-medical progression-readiness input for human-reviewed Master Gate decisions.',
        'limitations': [
            'Not a medical or injury assessment.',
            'Not an autonomous eligibility decision.',
            'Performance may drift as GymRAVANA members and trainer practices change.',
            'Trainer-recorded targets may contain human judgment bias.',
        ],
    }
    schema_path.write_text(json.dumps(feature_schema, indent=2), encoding='utf-8')
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    importance_path.write_text(json.dumps({'global_permutation_importance': importance_records, 'local_examples': local_explanations}, indent=2), encoding='utf-8')
    metadata_path.write_text(json.dumps(model_metadata, indent=2), encoding='utf-8')
    exported_paths = [model_path, schema_path, metrics_path, metadata_path, importance_path]
    print('Exported final local artifacts:')
    for path in exported_paths:
        print(f'- {path}')
else:
    print('No artifacts exported. Existing artifacts, if any, were not modified.')

## Integration boundary

Artifact export does not activate AI in Laravel. The next phase is a separately tested local inference service that validates the feature schema and model fingerprint, plus a Laravel client that fails safely whenever that service or a compatible reviewed artifact is unavailable.